In [1]:
import requests

ollama_api = "http://localhost:11434/api/generate"

In [2]:
# requests.post("http://localhost:11434/api/pull", json={"model": "glm-4.7-flash"})
# requests.post("http://localhost:11434/api/pull", json={"model": "qwen3:1.7b"})

In [ ]:
res = requests.post(
    ollama_api,
    json={
        "model": "qwen2.5:1.5b",
        "prompt": "hi! I have the following text snippet: 'The sky is blue'. Please answer with only a literal 'yes' or 'no': does the snippet refer to any color?",
        "think": False,
        "stream": False,
    },
)
res.status_code

200

In [12]:
res.json()["response"]

'yes'

In [3]:
import pandas as pd

laion_metadata = pd.read_csv("/home/jp/mgr2024/freesound_meta.csv")
laion_metadata

,audio_filename,caption1,caption2,freesound_id,username,freesound_url,license
0,train/91671.flac,small group laugh 6.,A group of about twenty people laughing during...,30043,tim.kahn,https://freesound.org/s/30043/,https://creativecommons.org/licenses/by/4.0/
1,train/102050.flac,mushroom disco 2.,weird loop.,103163,fons,https://freesound.org/s/103163/,http://creativecommons.org/publicdomain/zero/1.0/
2,train/247195.flac,Page Turn 24.,31/36 of Various Book manipulations.,20258,Koops,https://freesound.org/s/20258/,https://creativecommons.org/licenses/by/4.0/
3,train/352235.flac,yawn soft.,Recording of a male yawn with a zoom H6 in a s...,404384,EskimoNeil,https://freesound.org/s/404384/,http://creativecommons.org/licenses/by-nc/3.0/
4,train/457574.flac,Fender Chroma Polaris - Mourning - D5 (Mournin...,Single note sampled from an analog synthesizer...,289166,modularsamples,https://freesound.org/s/289166/,http://creativecommons.org/publicdomain/zero/1.0/
...,...,...,...,...,...,...,...
516882,test/464571.flac,Eastbay-C-504R.,"Made with music program, fun to do;).",319481,visual,https://freesound.org/s/319481/,https://creativecommons.org/licenses/by/4.0/
516883,test/474610.flac,amazed 0 30q.,Typical movie-sound to provide the short momen...,136402,Setuniman,https://freesound.org/s/136402/,https://creativecommons.org/licenses/by-nc/4.0/
516884,test/499425.flac,"SD14x08Tama-VLP,TSn-CS-v06.","A 1-shot sample taken of a 14-inch diameter, 8...",144719,quartertone,https://freesound.org/s/144719/,https://creativecommons.org/licenses/by/4.0/
516885,test/475963.flac,breathehope - kitchen - 09.,From my Kitchen.,2135,smallsushi,https://freesound.org/s/2135/,http://creativecommons.org/licenses/sampling+/...


In [17]:
llm_ready_desc = laion_metadata["caption1"] + "; " + laion_metadata["caption2"]
llm_ready_desc

0         small group laugh 6.; A group of about twenty ...
1                            mushroom disco 2.; weird loop.
2         Page Turn 24.; 31/36 of Various Book manipulat...
3         yawn soft.; Recording of a male yawn with a zo...
4         Fender Chroma Polaris - Mourning - D5 (Mournin...
                                ...                        
516882    Eastbay-C-504R.; Made with music program, fun ...
516883    amazed 0 30q.; Typical movie-sound to provide ...
516884    SD14x08Tama-VLP,TSn-CS-v06.; A 1-shot sample t...
516885        breathehope - kitchen - 09.; From my Kitchen.
516886    Tham B 048.; Recording of a single stroke 'Tha...
Length: 516887, dtype: object

In [19]:
import requests
from rich.progress import track

laion_metadata["llm_ready_desc"] = llm_ready_desc
prompt_question = "does the sample contain a one-shot sound similar to a snare?"
prompt_template = "I have the following text description of a short sound recording: '{}'. Please answer with only a yes or no: " + prompt_question

def _classify_description(s: str) -> bool:
    res = requests.post(
        ollama_api,
        json={
            "model": "glm-4.7-flash",
            "prompt": prompt_template.format(s),
            "think": False,
            "stream": False,
        },
    )
    return res.json()["response"]

# laion_metadata[prompt_question] = laion_metadata["llm_ready_desc"].apply(_classify_description)
results = []
for desc in track(laion_metadata["llm_ready_desc"].values):
    response = _classify_description(desc)
    result = False
    match response:
        case "yes" | "Yes":
            result = True
        case "no" | "No":
            result = False
        case _:
            print(f"got unexpected response: {response}")
    results.append(result)
laion_metadata[prompt_question] = results
laion_metadata

/home/jp/miniconda3/envs/mgr/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

KeyboardInterrupt: 

In [ ]:
# len(results), sum(results)

(15736, 705)

In [ ]:
# temporary to save the unfinished GLM 4.7 computation
# import json

# with open("laion_glm4.7res.json", "w") as f:
#     json.dump(results, f)

In [4]:
import json

with open("laion_glm4.7res.json") as f:
    results = json.load(f)

alleged_snares = laion_metadata.iloc[:len(results)].loc[results]
alleged_snares

,audio_filename,caption1,caption2,freesound_id,username,freesound_url,license
26,train/451514.flac,SMG Sound Hand Snap 0067754217.,Sound by Stolting Media Group.,600827,stoltingmediagroup,https://freesound.org/s/600827/,https://creativecommons.org/licenses/by/4.0/
39,train/66292.flac,HOR Gore Celeri 11.,Snapping celery in my basement:).,493519,Globbie,https://freesound.org/s/493519/,http://creativecommons.org/publicdomain/zero/1.0/
59,train/253015.flac,walk on carpet.,na.,120921,rocky12345,https://freesound.org/s/120921/,http://creativecommons.org/licenses/by-nc/3.0/
66,train/276645.flac,tomtom ride snare 110bpm-17.,tomtom ride snare kick 110bpm.,419885,@realdavidfloat,https://freesound.org/s/419885/,https://creativecommons.org/licenses/by/4.0/
76,train/32730.flac,SMG Sound Hand Snap 0156501952.,Sound by Stolting Media Group.,600479,stoltingmediagroup,https://freesound.org/s/600479/,https://creativecommons.org/licenses/by/4.0/
...,...,...,...,...,...,...,...
15660,train/65977.flac,"SD13x03-Pearl-MHP,0Sn-CS-v03.","A 1-shot sample taken of a 13-inch diameter, 3...",150668,quartertone,https://freesound.org/s/150668/,https://creativecommons.org/licenses/by/4.0/
15667,train/351606.flac,manonboschard 2013 S3.,S3600 hzchanging the speed -21.,166891,iut_Paris8,https://freesound.org/s/166891/,https://creativecommons.org/licenses/by-nc/4.0/
15691,train/203698.flac,090405 00 road bus practise cymbals.,across the road ppl practising .,76147,dkustic,https://freesound.org/s/76147/,http://creativecommons.org/licenses/by/3.0/
15692,train/255968.flac,"SD13x03-Pearl-MLP,TSn-HdC-v01.","A 1-shot sample taken of a 13-inch diameter, 3...",200221,quartertone,https://freesound.org/s/200221/,https://creativecommons.org/licenses/by/4.0/


In [6]:
laion_qwen25_snares = pd.read_csv("laion_snares.csv")
laion_metadata["qwen25_is_snare"] = laion_qwen25_snares
laion_metadata

,audio_filename,caption1,caption2,freesound_id,username,freesound_url,license,qwen25_is_snare
0,train/91671.flac,small group laugh 6.,A group of about twenty people laughing during...,30043,tim.kahn,https://freesound.org/s/30043/,https://creativecommons.org/licenses/by/4.0/,False
1,train/102050.flac,mushroom disco 2.,weird loop.,103163,fons,https://freesound.org/s/103163/,http://creativecommons.org/publicdomain/zero/1.0/,False
2,train/247195.flac,Page Turn 24.,31/36 of Various Book manipulations.,20258,Koops,https://freesound.org/s/20258/,https://creativecommons.org/licenses/by/4.0/,False
3,train/352235.flac,yawn soft.,Recording of a male yawn with a zoom H6 in a s...,404384,EskimoNeil,https://freesound.org/s/404384/,http://creativecommons.org/licenses/by-nc/3.0/,False
4,train/457574.flac,Fender Chroma Polaris - Mourning - D5 (Mournin...,Single note sampled from an analog synthesizer...,289166,modularsamples,https://freesound.org/s/289166/,http://creativecommons.org/publicdomain/zero/1.0/,False
...,...,...,...,...,...,...,...,...
516882,test/464571.flac,Eastbay-C-504R.,"Made with music program, fun to do;).",319481,visual,https://freesound.org/s/319481/,https://creativecommons.org/licenses/by/4.0/,False
516883,test/474610.flac,amazed 0 30q.,Typical movie-sound to provide the short momen...,136402,Setuniman,https://freesound.org/s/136402/,https://creativecommons.org/licenses/by-nc/4.0/,False
516884,test/499425.flac,"SD14x08Tama-VLP,TSn-CS-v06.","A 1-shot sample taken of a 14-inch diameter, 8...",144719,quartertone,https://freesound.org/s/144719/,https://creativecommons.org/licenses/by/4.0/,True
516885,test/475963.flac,breathehope - kitchen - 09.,From my Kitchen.,2135,smallsushi,https://freesound.org/s/2135/,http://creativecommons.org/licenses/sampling+/...,False


In [8]:
laion_metadata["qwen25_is_snare"].sum()

23622

In [4]:
import torchaudio as ta
import IPython
from IPython.display import display

# audio_dir = ds_dir / 'audio'

# def display_file(id: str) -> None:
#     audio, sr = ta.load(str(audio_dir / f"{id}.wav"))
#     display(IPython.display.Audio(audio, rate=sr))


# for _, s in labels_df[(labels_df["duration"] >= 0.1) & (labels_df["duration"] <= 0.3)].sample(20).iterrows():
#     print(s["word"], round(s["duration"], 4), s["original_text"])
    # display_file(s['sample_id'])

# laion_metadata[laion_metadata["qwen25_is_snare"]].sample(20)

In [5]:
laion_qwen25_snares = pd.read_csv("laion_snares_qwen25_14b.csv")
laion_metadata["qwen2514b_is_snare"] = laion_qwen25_snares
laion_metadata

,audio_filename,caption1,caption2,freesound_id,username,freesound_url,license,qwen2514b_is_snare
0,train/91671.flac,small group laugh 6.,A group of about twenty people laughing during...,30043,tim.kahn,https://freesound.org/s/30043/,https://creativecommons.org/licenses/by/4.0/,False
1,train/102050.flac,mushroom disco 2.,weird loop.,103163,fons,https://freesound.org/s/103163/,http://creativecommons.org/publicdomain/zero/1.0/,False
2,train/247195.flac,Page Turn 24.,31/36 of Various Book manipulations.,20258,Koops,https://freesound.org/s/20258/,https://creativecommons.org/licenses/by/4.0/,False
3,train/352235.flac,yawn soft.,Recording of a male yawn with a zoom H6 in a s...,404384,EskimoNeil,https://freesound.org/s/404384/,http://creativecommons.org/licenses/by-nc/3.0/,False
4,train/457574.flac,Fender Chroma Polaris - Mourning - D5 (Mournin...,Single note sampled from an analog synthesizer...,289166,modularsamples,https://freesound.org/s/289166/,http://creativecommons.org/publicdomain/zero/1.0/,False
...,...,...,...,...,...,...,...,...
516882,test/464571.flac,Eastbay-C-504R.,"Made with music program, fun to do;).",319481,visual,https://freesound.org/s/319481/,https://creativecommons.org/licenses/by/4.0/,False
516883,test/474610.flac,amazed 0 30q.,Typical movie-sound to provide the short momen...,136402,Setuniman,https://freesound.org/s/136402/,https://creativecommons.org/licenses/by-nc/4.0/,False
516884,test/499425.flac,"SD14x08Tama-VLP,TSn-CS-v06.","A 1-shot sample taken of a 14-inch diameter, 8...",144719,quartertone,https://freesound.org/s/144719/,https://creativecommons.org/licenses/by/4.0/,True
516885,test/475963.flac,breathehope - kitchen - 09.,From my Kitchen.,2135,smallsushi,https://freesound.org/s/2135/,http://creativecommons.org/licenses/sampling+/...,False


In [6]:
laion_metadata["qwen2514b_is_snare"].sum()

8560

In [7]:
laion_metadata[laion_metadata["qwen2514b_is_snare"]].sample(20)

,audio_filename,caption1,caption2,freesound_id,username,freesound_url,license,qwen2514b_is_snare
304265,train/230884.flac,snap4.,Snap of fingers.,235283,Godowan,https://freesound.org/s/235283/,http://creativecommons.org/publicdomain/zero/1.0/,True
64280,train/94681.flac,"SD14x08Tama-MLP,LSn-HdC-v04.","A 1-shot sample taken of a 14-inch diameter, 8...",143382,quartertone,https://freesound.org/s/143382/,https://creativecommons.org/licenses/by/4.0/,True
180785,train/331110.flac,dry snare center 06.,Snare drum recorded using a combination of dir...,382785,pjcohen,https://freesound.org/s/382785/,https://creativecommons.org/licenses/by/4.0/,True
293229,train/339320.flac,SBUC SNARE 003.,"Real and sampled snares phase-matched, layered...",38935,sandyrb,https://freesound.org/s/38935/,https://creativecommons.org/licenses/by/4.0/,True
12247,train/113412.flac,steel snare 9.,Sounds of a steel-shell snare drum.,624203,strangehorizon,https://freesound.org/s/624203/,http://creativecommons.org/publicdomain/zero/1.0/,True
394104,train/23910.flac,Snare.,Snare Sample.,78227,Spol,https://freesound.org/s/78227/,http://creativecommons.org/licenses/by/3.0/,True
124760,train/274828.flac,SMG Sound Drum Snare 0592949721.,Drum Sound by Stolting Media Group.,598403,stoltingmediagroup,https://freesound.org/s/598403/,https://creativecommons.org/licenses/by/4.0/,True
504825,test/491902.flac,PPU-SNARE - Marker #49.,"Heavily processed, strongly compressed, tight ...",208395,crispydinner,https://freesound.org/s/208395/,http://creativecommons.org/publicdomain/zero/1.0/,True
40693,train/13414.flac,DR Ruiner Snare 1.,snare_1 one-hit wav samples of a circuit bent ...,18192,Roil Noise,https://freesound.org/s/18192/,http://creativecommons.org/publicdomain/zero/1.0/,True
417896,train/145262.flac,"SD14x08Tama-LP,TSn-HdC-v11.","A 1-shot sample taken of a 14-inch diameter, 8...",143686,quartertone,https://freesound.org/s/143686/,https://creativecommons.org/licenses/by/4.0/,True
